#### q1: total customers who placed orders and distribution across states

In [ ]:
-- total number of customers who have placed orders with distribution of the customers across states.
select
    coalesce(c.state, 'Grand Total') as state,
    count(distinct o.customer_id) as customers_with_orders
from newwheels_order_t o
join newwheels_customer_t c
    on o.customer_id = c.customer_id
group by c.state with rollup
order by
    c.state is not null,
    customers_with_orders desc;

#### q2: top 5 most preferred vehicle makers

In [ ]:
-- top 5 vehicle makers with the highest number of customers
select p.vehicle_maker,
       count(distinct o.customer_id) as customer_count
from newwheels_order_t o
join newwheels_product_t p on o.product_id = p.product_id
group by p.vehicle_maker
order by customer_count desc
limit 5;

#### q3: most preferred vehicle maker in each state using rank

In [ ]:
-- top vehicle maker in each state based on the number of customers
with maker_rank as (
    select 
        c.state,
        p.vehicle_maker,
        count(distinct o.customer_id) as customer_count,
        dense_rank() over (
            partition by c.state
            order by count(distinct o.customer_id) desc
        ) as rnk
    from newwheels_order_t o
    join newwheels_customer_t c 
        on o.customer_id = c.customer_id
    join newwheels_product_t p 
        on o.product_id = p.product_id
    group by c.state, p.vehicle_maker
)

select state, vehicle_maker, customer_count
from maker_rank
where rnk = 1
order by customer_count desc, state;

#### q4: overall average rating and average rating per quarter

In [ ]:
-- overall average rating given by the customers and average rating per quarter
select
    round(avg(field(lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good')), 1) as overall_avg_rating,

    round(avg(case when quarter_number = 1 then
        field(lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good') end), 1) as q1_avg,

    round(avg(case when quarter_number = 2 then
        field(lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good') end), 1) as q2_avg,

    round(avg(case when quarter_number = 3 then
        field(lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good') end), 1) as q3_avg,

    round(avg(case when quarter_number = 4 then
        field(lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good') end), 1) as q4_avg
from newwheels_order_t;

#### q5: percentage distribution of feedback and dissatisfaction trend

In [ ]:
-- percentage distribution of feedback from the customers and how it has changed over the quarters
select
    case
        when quarter_number is null then 'Overall'
        else concat('Q', quarter_number)
    end as period,
    round(100 * avg(lower(customer_feedback) = 'very bad'), 1) as pct_very_bad,
    round(100 * avg(lower(customer_feedback) = 'bad'), 1) as pct_bad,
    round(100 * avg(lower(customer_feedback) = 'okay'), 1) as pct_okay,
    round(100 * avg(lower(customer_feedback) = 'good'), 1) as pct_good,
    round(100 * avg(lower(customer_feedback) = 'very good'), 1) as pct_very_good
from newwheels_order_t
group by quarter_number with rollup
order by quarter_number is not null, quarter_number;

#### q6: trend of number of orders by quarter

In [ ]:
-- total number of orders placed in each quarter
select
    coalesce(concat('Q', quarter_number), 'Overall') as quarter,
    count(distinct order_id) as total_orders
from newwheels_order_t
group by quarter_number with rollup
order by quarter_number is not null, quarter_number;


#### q7: net revenue generated by the company and quarter-over-quarter % change in net revenue

In [ ]:
-- net revenue generated in each quarter and the quarter-on-quarter percentage change in net revenue
with quarterly as (
    select
        quarter_number,
        sum(quantity * vehicle_price * (1 - discount / 100.0)) as net_revenue,
        lag(sum(quantity * vehicle_price * (1 - discount / 100.0)))
            over (order by quarter_number) as previous_revenue
    from newwheels_order_t
    group by quarter_number
)

select
    'Overall' as period,
    round(sum(quantity * vehicle_price * (1 - discount / 100.0)), 2) as net_revenue,
    null as qoq_pct_change
from newwheels_order_t

union all

select
    concat('Q', quarter_number),
    round(net_revenue, 2),
    round((net_revenue - previous_revenue) / previous_revenue * 100, 1)
from quarterly;

#### q8: trend of net revenue and number of orders by quarter

In [ ]:
-- net revenue and total orders per quarter
select
    concat('Q', quarter_number) as quarter,
    round(sum(quantity * vehicle_price * (1 - discount / 100.0)),2 ) as net_revenue,
    count(distinct order_id) as total_orders
from newwheels_order_t
group by quarter_number
order by quarter_number;

#### q9: average discount offered for different types of credit cards? 

In [ ]:
-- average discount offered to customers based on their credit card type
select
    c.credit_card_type,
    round(avg(o.discount), 2) as avg_discount
from newwheels_order_t o
join newwheels_customer_t c
    on o.customer_id = c.customer_id
group by c.credit_card_type
order by avg_discount desc;

#### q10: average time taken to ship the placed orders for each quarter? 

In [ ]:
-- average shipping time (in days) per quarter
select
    coalesce(concat('Q', quarter_number), 'Overall') as quarter,
    round(avg(datediff(ship_date, order_date)), 1) as avg_shipping_days
from newwheels_order_t
group by quarter_number with rollup
order by quarter_number is not null, quarter_number;

#### Overall

In [ ]:
select
    round(sum(quantity * vehicle_price * (1 - discount / 100.0)), 2) as total_revenue,
    count(distinct order_id) as total_orders,
    count(distinct customer_id) as total_customers,

    round(avg(field(
        lower(customer_feedback),
        'very bad', 'bad', 'okay', 'good', 'very good'
    )), 1) as avg_rating,

    round(
        100 * avg(lower(customer_feedback) in ('good', 'very good')),
        1
    ) as pct_good_feedback,

    round(avg(datediff(ship_date, order_date)), 1) as avg_days_to_ship,

    round(sum(
        case when quarter_number = 4
        then quantity * vehicle_price * (1 - discount / 100.0)
        end
    ), 2) as last_quarter_revenue,

    count(distinct
        case when quarter_number = 4 then order_id end
    ) as last_quarter_orders

from newwheels_order_t;